In [ ]:
# Binary verifier v2: positional indexing plus the DDI-2013 annotation rules.
#
# v1 scored NONE recall 0.532 (excluding same-entity pairs). Reading the false positives
# showed most were unanswerable as posed: pairs were named by surface form, so a sentence
# with two aripiprazole mentions asked the same question twice with two different correct
# answers. Rule P1 says only the mention tied to the interaction participates.
#
# Cell 1 sizes that problem from data already on disk. Cell 2 checks the new rendering.
# Cell 4 is the decision point.

import json
import importlib
from collections import Counter

import ddi.verify_binary
importlib.reload(ddi.verify_binary)

from ddi.data import build_human
from ddi.synth import RAW
from ddi.verify_binary import (build_batches, render, make_binary_verifier,
                               load_verdicts, calibration_report)

train, dev, val = build_human()


# ===========================================================================
# Cell 1: how much of v1's error was unanswerable? Free, no worker.
# A pair is ambiguous if the same surface-form pair appears twice in one sentence with
# different gold labels: v1 could not have got both right.
# ===========================================================================
amb = tot = 0
amb_examples = []
for line in (RAW / "verify-bin-humandev.jsonl").read_text().splitlines():
    if not line:
        continue
    r = json.loads(line)
    if r.get("error") or not r.get("sample"):
        continue
    seen = {}
    for p, g in zip(r["spec"]["pairs"], r["spec"]["gold"]):
        k = tuple(sorted(x.lower() for x in p))
        tot += 1
        if k in seen and seen[k] != g:
            amb += 1
            if len(amb_examples) < 5:
                amb_examples.append((k, r["spec"]["text"][:200]))
        seen[k] = g

print(f"{amb}/{tot} pairs unanswerable by surface form = {amb / max(tot, 1):.3f}\n")
for k, t in amb_examples:
    print(f"  {k}\n    {t}\n")



In [ ]:
import re

# ===========================================================================
# Cell 2: does the new rendering fix it? Also free.
# ===========================================================================
batches = build_batches(dev)
print(f"{len(dev)} instances -> {len(batches)} sentences")
print(f"pairs per batch: mean {sum(len(b['pairs']) for b in batches) / len(batches):.1f}, "
      f"max {max(len(b['pairs']) for b in batches)}")

# same ambiguity test, now on mention indices rather than surface forms
amb2 = tot2 = 0
for b in batches:
    seen = {}
    for p, g in zip(b["pairs"], b["gold"]):
        k = tuple(sorted(p))
        tot2 += 1
        if k in seen and seen[k] != g:
            amb2 += 1
        seen[k] = g
print(f"\nafter indexing: {amb2}/{tot2} unanswerable = {amb2 / max(tot2, 1):.4f}")
print("should be 0.0000; anything else means _marked_spans is misaligned")

# read the aripiprazole case specifically, it is the one that broke v1
for b in batches:
    if "aripiprazole" in b["text"].lower() and b["n_mentions"] >= 3:
        print("\n" + render(b))
        print("gold:", b["gold"])
        break

# and one with a leading title, which is rule P6
for b in batches:
    if re.match(r"^[A-Z][A-Za-z\- ]+\[\d+\]\s*:", b["text"]):
        print("\n" + render(b)[:800])
        print("gold:", b["gold"])
        break


In [ ]:


# ===========================================================================
# Cell 3: cap the enumeration sentences and confirm what is dropped
# ===========================================================================
MAX_PAIRS = 30
kept = [b for b in batches if len(b["pairs"]) <= MAX_PAIRS]
print(f"{len(kept)} batches after capping at {MAX_PAIRS} pairs "
      f"({len(batches) - len(kept)} dropped, "
      f"{sum(len(b['pairs']) for b in kept)} pairs)")
# the dropped ones are the flattened tables and bare drug lists, which are excluded from
# the training pool anyway



In [ ]:

# ===========================================================================
# Cell 4: calibrate. ~10 min with a live worker. THE DECISION POINT.
# ===========================================================================
import os
from openai import OpenAI
from ddi.synth import generate_raw

MODEL, API, EFFORT = "gpt-oss-120b", "responses", "high"
GEN = "verify-bin2-humandev-3"

client = OpenAI(base_url="http://api.llm.apps.os.dcs.gla.ac.uk/v1",
                api_key=os.environ["IDA_LLM_API_KEY"], max_retries=5, timeout=180.0)
verify_fn = make_binary_verifier(client, model=MODEL, reasoning_effort=EFFORT, api=API)

generate_raw(kept, verify_fn, gen_id=GEN, max_workers=16)

d = load_verdicts(GEN)
same = d.e1.str.lower().str.rstrip("s") == d.e2.str.lower().str.rstrip("s")
print(f"\n{same.sum()} same-entity pairs ({same.mean():.3f}), "
      f"{d[same].flagged.mean():.3f} flagged")

print("\nall pairs")
cal = calibration_report(d)
print("\nexcluding same-entity pairs")
cal2 = calibration_report(d[~same])

print(f"""
v1 was 0.619 / 0.532. Five-way verifier 0.81.
PASS if NONE recall >= 0.95 excluding same-entity pairs.
""")


In [ ]:
from ddi.verify_binary import load_verdicts, calibration_report
d = load_verdicts("verify-bin2-humandev")
same = d.e1.str.lower().str.rstrip("s") == d.e2.str.lower().str.rstrip("s")
cal2 = calibration_report(d[~same])

In [ ]:
# ===========================================================================
# Cell 5: what is left wrong. Read these before deciding anything.
# ===========================================================================
dd = d[~same]
fp = dd[(~dd.gold_pos) & dd.flagged]
fn = dd[dd.gold_pos & (~dd.flagged)]
print(f"{len(fp)} false flags, {len(fn)} missed\n")
 
# split by whether the sentence has repeated mentions, which is where P1 bites
rep = fp[fp.n_mentions > fp.groupby("sent_id").m1.transform("nunique") + 1]
print(f"false flags in sentences with repeated mentions: {len(rep)}\n")
 
print("--- flagged, gold NONE ---")
for _, r in fp.head(12).iterrows():
    print(f"[{r.m1} {r.e1} / {r.m2} {r.e2}]\n  {r.text[:260]}\n")
 
print("--- missed, gold positive ---")
for _, r in fn.head(8).iterrows():
    print(f"[{r.gold}: {r.m1} {r.e1} / {r.m2} {r.e2}]\n  {r.text[:260]}\n")

In [ ]:


# ===========================================================================
# Cell 6: only if cell 4 passed. Measure drift in v14 and v15.
# ~5,800 batches each.
# ===========================================================================
from ddi.manifest import load_dataset

V14_ID = "20260807-123340-ff79db"
V15_ID = "20260816-005908-3d6539"

for name, ds_id in [("v14", V14_ID), ("v15", V15_ID)]:
    inst, _ = load_dataset(ds_id)
    b = [x for x in build_batches(inst) if len(x["pairs"]) <= MAX_PAIRS]
    print(f"{name}: {len(b)} sentences, {sum(len(x['pairs']) for x in b)} pairs")
    generate_raw(b, verify_fn, gen_id=f"verify-bin2-{name}", max_workers=16)

    v = load_verdicts(f"verify-bin2-{name}")
    none = v[~v.gold_pos]
    raw = none.flagged.mean()
    fpr = 1 - cal2["none_recall"]
    corrected = max((raw - fpr) / max(cal2["pos_recall"] - fpr, 1e-9), 0.0)
    print(f"  raw flag rate {raw:.4f}   corrected {corrected:.4f}\n")

print("""
The corrected rate estimates the fraction of NONE labels that are wrong. The hypothesis
is that v15 is higher, because removing per-drug role marking removed the constraint
that made NONE safe by construction. If both are low and similar, label noise does not
explain the 0.046 regression and it stays unexplained.
""")